<a href="https://colab.research.google.com/github/MaGiaVy/DoAnPython/blob/main/project/notebooks/Nhom3thangcuti_Tuan4/notebooks/Tuan4_GiaVy_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏆 Baseline 2: MobileCLIP (Zero-Shot Multimodal)
## E-commerce Visual Search — Shopee Dataset (34,250 items) | Google Colab GPU

**Pipeline:**
- 🍎 **Model:** `MobileCLIP` (Apple) — Image + Text cùng embedding space
- 🔀 **Fusion:** `L2_Norm(α × img_feat + (1-α) × txt_feat)`
- 🔍 **Search:** FAISS `IndexFlatIP` (Cosine Similarity)
- 📊 **Metrics:** mAP@5, Precision@1, Recall@5

**Fallback:** Nếu MobileCLIP không cài được → tự động dùng `openai/clip-vit-base-patch32` (HuggingFace).

**Dataset Split (STRICT — NO DATA LEAKAGE):**
- Gallery : toàn bộ 34,250 ảnh
- Val queries (20%) : ~6,850 → grid search `alpha`
- Test queries (80%) : ~27,400 → đánh giá cuối, chạy **1 lần duy nhất**

## ⚙️ Cell 0: Kiểm tra GPU & Runtime

In [1]:
# Kiểm tra GPU — nếu thấy 'No GPU' hãy vào Runtime > Change runtime type > T4 GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU khả dụng:')
    print(result.stdout)
else:
    print('❌ Không tìm thấy GPU!')
    print('👉 Vào Runtime > Change runtime type > chọn T4 GPU rồi thử lại!')

import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name        : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

✅ GPU khả dụng:
Fri May 29 02:57:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------------------------

## 📦 Cell 1: Cài đặt thư viện & MobileCLIP

In [2]:
import sys, subprocess

# Thư viện cơ bản
!pip install -q faiss-gpu timm

# ─── Thử cài MobileCLIP từ Apple ─────────────────────────────────────────────
USE_MOBILECLIP = False
print('⏳ Đang thử cài MobileCLIP (Apple)...')
try:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'git+https://github.com/apple/ml-mobileclip.git'],
        capture_output=True, text=True, timeout=180
    )
    import mobileclip
    USE_MOBILECLIP = True
    print('✅ MobileCLIP (Apple) cài thành công!')
except Exception as e:
    print(f'⚠️  Không cài được MobileCLIP: {e}')
    print('🔄 Fallback → openai/clip-vit-base-patch32 (HuggingFace)')
    !pip install -q transformers

print(f'\n🔧 Chế độ: {"MobileCLIP (Apple)" if USE_MOBILECLIP else "CLIP HuggingFace Fallback"}')

ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu
⏳ Đang thử cài MobileCLIP (Apple)...


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


✅ MobileCLIP (Apple) cài thành công!

🔧 Chế độ: MobileCLIP (Apple)


## 📂 Cell 2: Kết nối Google Drive & Tự động dò tìm đường dẫn Dataset

In [3]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# ─── TỰ ĐỘNG DÒ TÌM ĐƯỜNG DẪN DỮ LIỆU PHÙ HỢP ───────────────────────────────
POSSIBLE_PATHS = [
    '/content/drive/MyDrive/DoAnPython/DuLieuPython',
    '/content/drive/MyDrive/DuLieuPython',
    '/content/drive/My Drive/DoAnPython/DuLieuPython',
    '/content/drive/My Drive/DuLieuPython'
]

DATA_DIR = None
for path in POSSIBLE_PATHS:
    if os.path.exists(os.path.join(path, 'train.csv')):
        DATA_DIR = path
        break

if DATA_DIR is None:
    # Fallback mặc định
    DATA_DIR = '/content/drive/MyDrive/DuLieuPython/DuLieuPython'
    print(f'⚠️ Không tìm thấy đường dẫn có chứa train.csv. Dùng mặc định: {DATA_DIR}')
else:
    print(f'✅ Đã tự động phát hiện thư mục dữ liệu tại: {DATA_DIR}')

CSV_PATH = os.path.join(DATA_DIR, 'train.csv')
IMAGE_ZIP_PATH = os.path.join(DATA_DIR, 'train_images')

# Thư mục giải nén cục bộ trên Colab để đọc ảnh siêu nhanh
EXTRACTED_DIR = '/content/train_images_extracted'
IMG_DIR = os.path.join(EXTRACTED_DIR, 'train_images')

# Giải nén file zip ảnh vào thư mục cục bộ của Colab nếu chưa giải nén
if not os.path.exists(IMG_DIR):
    if os.path.exists(IMAGE_ZIP_PATH):
        print('⏳ Đang giải nén train_images.zip vào Colab (sẽ mất khoảng 1-2 phút)...')
        !unzip -q {IMAGE_ZIP_PATH} -d {EXTRACTED_DIR}
        print('✅ Giải nén thành công!')
    else:
        print(f'❌ Không tìm thấy file zip tại {IMAGE_ZIP_PATH}. Vui lòng kiểm tra lại Drive!')
else:
    print('✅ Đã có thư mục ảnh giải nén cục bộ!')

# Kiểm tra cuối cùng
for name, p in [('File CSV', CSV_PATH), ('Thư mục ảnh giải nén', IMG_DIR)]:
    status = 'Đã sẵn sàng' if os.path.exists(p) else 'KHÔNG tìm thấy – kiểm tra lại!'
    print(f'   {name}: {status} ({p})')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Đã tự động phát hiện thư mục dữ liệu tại: /content/drive/MyDrive/DuLieuPython
✅ Đã có thư mục ảnh giải nén cục bộ!
   File CSV: Đã sẵn sàng (/content/drive/MyDrive/DuLieuPython/train.csv)
   Thư mục ảnh giải nén: Đã sẵn sàng (/content/train_images_extracted/train_images)


## 🔧 Cell 3: Import & Cấu hình

In [4]:
# Install faiss-cpu as faiss-gpu installation failed previously.
!pip install -q faiss-cpu

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import faiss

# ─── Cấu hình ────────────────────────────────────────────────────────────────
MOBILECLIP_VARIANT = 'mobileclip_s0'          # s0 (nhanh nhất) / s1 / s2 / b
MOBILECLIP_CKPT    = '/tmp/mobileclip_s0.pt'  # sẽ download nếu cần

BATCH_SIZE   = 128    # T4 GPU ~16GB VRAM
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
RANDOM_SEED  = 42
NUM_WORKERS  = 2

print(f'✅ Thiết bị        : {DEVICE}')
print(f'✅ Batch size      : {BATCH_SIZE}')

✅ Thiết bị        : cuda
✅ Batch size      : 128


## 📊 Cell 4: Đọc dữ liệu & Chia tập (STRICT SPLIT)

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(CSV_PATH)
print(f'📊 Tổng số mẫu : {len(df):,}')
print(f'📋 Các cột     : {list(df.columns)}')
print(df.head(3))

# ─── KHÔNG stratify vì số lớp (11,014) > kích thước validation (6,850) ───
df_gallery = df.copy()

val_idx, test_idx = train_test_split(
    df.index.tolist(),
    test_size    = 0.8,
    random_state = RANDOM_SEED
)

df_val  = df.loc[val_idx].reset_index(drop=True)
df_test = df.loc[test_idx].reset_index(drop=True)

print(f'\n🗂️  Gallery size  : {len(df_gallery):,} ảnh (toàn bộ dataset)')
print(f'✅ Val queries   : {len(df_val):,} ảnh  → grid search alpha')
print(f'✅ Test queries  : {len(df_test):,} ảnh  → đánh giá cuối (1 lần!)')

📊 Tổng số mẫu : 34,250
📋 Các cột     : ['posting_id', 'image', 'image_phash', 'title', 'label_group']
         posting_id                                 image       image_phash  \
0   train_129225211  0000a68812bc7e98c42888dfb1c07da0.jpg  94974f937d4c2433   
1  train_3386243561  00039780dfc94d01db8676fe789ecd05.jpg  af3f9460c2838f0f   
2  train_2288590299  000a190fdd715a2a36faed16e2c65df7.jpg  b94cb00ed3e50f78   

                                               title  label_group  
0                          Paper Bag Victoria Secret    249114794  
1  Double Tape 3M VHB 12 mm x 4,5 m ORIGINAL / DO...   2937985045  
2        Maling TTS Canned Pork Luncheon Meat 397 gr   2395904891  

🗂️  Gallery size  : 34,250 ảnh (toàn bộ dataset)
✅ Val queries   : 6,850 ảnh  → grid search alpha
✅ Test queries  : 27,400 ảnh  → đánh giá cuối (1 lần!)


## 🍎 Cell 5: Tải MobileCLIP (hoặc Fallback CLIP)

In [6]:
if USE_MOBILECLIP:
    # ─── MobileCLIP (Apple) ───────────────────────────────────────────────────
    import mobileclip, urllib.request

    CKPT_URLS = {
        'mobileclip_s0': 'https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_s0.pt',
        'mobileclip_s1': 'https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_s1.pt',
        'mobileclip_s2': 'https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_s2.pt',
        'mobileclip_b' : 'https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_b.pt',
    }

    if not os.path.exists(MOBILECLIP_CKPT):
        print(f'⏬ Đang download checkpoint {MOBILECLIP_VARIANT}...')
        urllib.request.urlretrieve(CKPT_URLS[MOBILECLIP_VARIANT], MOBILECLIP_CKPT)
        print(f'✅ Đã lưu: {MOBILECLIP_CKPT}')

    print(f'⏳ Đang tải {MOBILECLIP_VARIANT}...')
    clip_model, _, preprocess = mobileclip.create_model_and_transforms(
        MOBILECLIP_VARIANT, pretrained=MOBILECLIP_CKPT
    )
    tokenizer = mobileclip.get_tokenizer(MOBILECLIP_VARIANT)
    clip_model = clip_model.to(DEVICE).eval()

    with torch.no_grad():
        _dummy = torch.randn(1, 3, 256, 256).to(DEVICE)
        embed_dim = clip_model.encode_image(_dummy).shape[-1]

    MODEL_LABEL = f'MobileCLIP ({MOBILECLIP_VARIANT})'

else:
    # ─── Fallback: HuggingFace CLIP ───────────────────────────────────────────
    from transformers import CLIPModel, CLIPProcessor

    HF_MODEL = 'openai/clip-vit-base-patch32'
    print(f'⏳ Đang tải {HF_MODEL}...')
    clip_model = CLIPModel.from_pretrained(HF_MODEL).to(DEVICE).eval()
    preprocess = CLIPProcessor.from_pretrained(HF_MODEL)
    tokenizer  = None
    embed_dim  = clip_model.config.projection_dim

    MODEL_LABEL = f'CLIP ({HF_MODEL})'

print(f'\n✅ {MODEL_LABEL} đã sẵn sàng')
print(f'📐 Embedding dim: {embed_dim}')

⏳ Đang tải mobileclip_s0...

✅ MobileCLIP (mobileclip_s0) đã sẵn sàng
📐 Embedding dim: 512


## 🖼️📝 Cell 6: Hàm trích xuất Image & Text Features

In [7]:
class ShopeeImageDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, return_tensor=True):
        self.df            = df
        self.img_dir       = img_dir
        self.transform     = transform
        self.return_tensor = return_tensor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        fname = self.df.iloc[idx]['image']
        try:
            img = Image.open(os.path.join(self.img_dir, fname)).convert('RGB')
        except Exception:
            img = Image.new('RGB', (256, 256), (128, 128, 128))
        if self.return_tensor and self.transform:
            return self.transform(img)
        return img


@torch.no_grad()
def extract_image_features_clip(df_input, img_dir, batch_size=128, num_workers=2):
    all_feats = []

    if USE_MOBILECLIP:
        dataset = ShopeeImageDataset(df_input, img_dir,
                                     transform=preprocess, return_tensor=True)
        loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=True)
        for imgs in tqdm(loader, desc='🖼️ MobileCLIP image features'):
            feats = clip_model.encode_image(imgs.to(DEVICE))
            # ✅ FIX: normalize để scale đồng nhất với text features
            feats = feats / feats.norm(dim=-1, keepdim=True)
            all_feats.append(feats.cpu().float().numpy())
    else:
        dataset = ShopeeImageDataset(df_input, img_dir, return_tensor=False)
        for i in tqdm(range(0, len(dataset), batch_size),
                      desc='🖼️ CLIP HuggingFace image features'):
            batch  = [dataset[j] for j in range(i, min(i + batch_size, len(dataset)))]
            inputs = preprocess(images=batch, return_tensors='pt',
                                padding=True).to(DEVICE)
            feats  = clip_model.get_image_features(**inputs)
            # ✅ FIX: normalize
            feats = feats / feats.norm(dim=-1, keepdim=True)
            all_feats.append(feats.cpu().float().numpy())

    return np.vstack(all_feats)


@torch.no_grad()
def extract_text_features_clip(df_input, batch_size=256):
    titles    = df_input['title'].fillna('').tolist()
    all_feats = []

    if USE_MOBILECLIP:
        for i in tqdm(range(0, len(titles), batch_size),
                      desc='📝 MobileCLIP text features'):
            tokens = tokenizer(titles[i:i + batch_size]).to(DEVICE)
            feats  = clip_model.encode_text(tokens)
            feats = feats / feats.norm(dim=-1, keepdim=True)
            all_feats.append(feats.cpu().float().numpy())
    else:
        for i in tqdm(range(0, len(titles), batch_size),
                      desc='📝 CLIP HuggingFace text features'):
            inputs = preprocess(
                text=titles[i:i + batch_size], return_tensors='pt',
                padding=True, truncation=True, max_length=77
            ).to(DEVICE)
            feats  = clip_model.get_text_features(**inputs)
            # ✅ FIX: normalize (nhánh MobileCLIP đã có, HuggingFace chưa có)
            feats = feats / feats.norm(dim=-1, keepdim=True)
            all_feats.append(feats.cpu().float().numpy())

    return np.vstack(all_feats)


print('✅ Hàm trích xuất features đã sẵn sàng')

## 🔀 Cell 7: Fusion, FAISS & Metrics Utils

In [8]:
def fuse_and_normalize_clip(img_feats, txt_feats, alpha):
    fused = alpha * img_feats + (1 - alpha) * txt_feats
    norms = np.linalg.norm(fused, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1e-10, norms)
    return (fused / norms).astype(np.float32)


def build_faiss_index(features):
    index = faiss.IndexFlatIP(features.shape[1])
    index.add(features)
    return index

def get_ground_truth_dict(df_input):
    gt = {}
    for _, grp in df_input.groupby('label_group'):
        ids = set(grp['posting_id'].tolist())
        for pid in ids:
            gt[pid] = ids
    return gt


def evaluate_retrieval_clip(query_df, gallery_df, query_img, query_txt,
                             gallery_img, gallery_txt, alpha, K=5):
    q_fused      = fuse_and_normalize_clip(query_img, query_txt, alpha)
    g_fused      = fuse_and_normalize_clip(gallery_img, gallery_txt, alpha)
    gt_dict      = get_ground_truth_dict(gallery_df)
    index        = build_faiss_index(g_fused)
    _, indices   = index.search(q_fused, K + 1)
    gallery_pids = gallery_df['posting_id'].tolist()

    ap_list, p1_list, r5_list = [], [], []

    for i, row in enumerate(query_df.itertuples()):
        qid      = row.posting_id
        relevant = gt_dict.get(qid, set()) - {qid}
        if not relevant:
            continue

        retrieved = []
        for idx in indices[i]:
            pid = gallery_pids[idx]
            if pid != qid:
                retrieved.append(pid)
            if len(retrieved) == K:
                break

        hits, ap = 0, 0.0
        for rank, pid in enumerate(retrieved, 1):
            if pid in relevant:
                hits += 1
                ap   += hits / rank
        ap_list.append(ap / min(len(relevant), K))
        p1_list.append(1.0 if (retrieved and retrieved[0] in relevant) else 0.0)
        r5_list.append(len(set(retrieved) & relevant) / len(relevant))

    return {
        'mAP@5'      : float(np.mean(ap_list)),
        'Precision@1': float(np.mean(p1_list)),
        'Recall@5'   : float(np.mean(r5_list)),
    }


print('✅ Hàm fusion / FAISS / evaluate đã sẵn sàng')

✅ Hàm fusion / FAISS / evaluate đã sẵn sàng


## 🔍 Cell 8: Trích xuất tất cả Features

In [9]:
# ─── Gallery ─────────────────────────────────────────────────────────────────
print('📦 Gallery image features...')
gallery_img_feats = extract_image_features_clip(df_gallery, IMG_DIR, BATCH_SIZE, NUM_WORKERS)
print('📦 Gallery text features...')
gallery_txt_feats = extract_text_features_clip(df_gallery)
print(f'✅ Gallery img: {gallery_img_feats.shape} | txt: {gallery_txt_feats.shape}')

# ─── Val ─────────────────────────────────────────────────────────────────────
print('\n📦 Val image features...')
val_img_feats = extract_image_features_clip(df_val, IMG_DIR, BATCH_SIZE, NUM_WORKERS)
print('📦 Val text features...')
val_txt_feats = extract_text_features_clip(df_val)
print(f'✅ Val img: {val_img_feats.shape} | txt: {val_txt_feats.shape}')

# ─── Test ────────────────────────────────────────────────────────────────────
print('\n📦 Test image features...')
test_img_feats = extract_image_features_clip(df_test, IMG_DIR, BATCH_SIZE, NUM_WORKERS)
print('📦 Test text features...')
test_txt_feats = extract_text_features_clip(df_test)
print(f'✅ Test img: {test_img_feats.shape} | txt: {test_txt_feats.shape}')

📦 Gallery image features...


🖼️ MobileCLIP image features:   0%|          | 0/268 [00:00<?, ?it/s]

📦 Gallery text features...


📝 MobileCLIP text features:   0%|          | 0/134 [00:00<?, ?it/s]

✅ Gallery img: (34250, 512) | txt: (34250, 512)

📦 Val image features...


...

✅ Val img: (6850, 512) | txt: (6850, 512)

📦 Test image features...


🖼️ MobileCLIP image features:   0%|          | 0/215 [00:00<?, ?it/s]

📦 Test text features...


📝 MobileCLIP text features:   0%|          | 0/108 [00:00<?, ?it/s]

✅ Test img: (27400, 512) | txt: (27400, 512)


## 🎯 Cell 9: Grid Search Alpha — Validation Set

In [10]:
# ⚠️ Grid search CHỈ trên VAL — KHÔNG đụng Test!

alphas = np.arange(0.1, 1.0, 0.1).round(1)
print(f'🔍 Grid search alpha ∈ {alphas.tolist()}')
print(f'   (alpha=1.0 → chỉ image | alpha=0.0 → chỉ text)')
print('─' * 64)

val_results_2 = []
best_alpha_2, best_map5_2 = None, -1.0

for alpha in alphas:
    m = evaluate_retrieval_clip(
        query_df    = df_val,
        gallery_df  = df_gallery,
        query_img   = val_img_feats,
        query_txt   = val_txt_feats,
        gallery_img = gallery_img_feats,
        gallery_txt = gallery_txt_feats,
        alpha=alpha, K=5
    )
    val_results_2.append({'alpha': alpha, **m})
    marker = ' ← best' if m['mAP@5'] > best_map5_2 else ''
    print(f'  α={alpha:.1f} | mAP@5={m["mAP@5"]:.4f} | '
          f'P@1={m["Precision@1"]:.4f} | R@5={m["Recall@5"]:.4f}{marker}')
    if m['mAP@5'] > best_map5_2:
        best_map5_2   = m['mAP@5']
        best_alpha_2  = alpha

print('─' * 64)
print(f'\n🏆 BEST_ALPHA_2 = {best_alpha_2:.1f}  (Val mAP@5 = {best_map5_2:.4f})')

# In bảng đầy đủ
df_val_summary = pd.DataFrame(val_results_2)
print('\n📊 Bảng val đầy đủ:')
print(df_val_summary.to_string(index=False))

🔍 Grid search alpha ∈ [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
   (alpha=1.0 → chỉ image | alpha=0.0 → chỉ text)
────────────────────────────────────────────────────────────────
  α=0.1 | mAP@5=0.6299 | P@1=0.6793 | R@5=0.6211 ← best
  α=0.2 | mAP@5=0.6654 | P@1=0.7124 | R@5=0.6507 ← best
  α=0.3 | mAP@5=0.7139 | P@1=0.7543 | R@5=0.6923 ← best
  α=0.4 | mAP@5=0.7548 | P@1=0.7841 | R@5=0.7260 ← best
  α=0.5 | mAP@5=0.7706 | P@1=0.7942 | R@5=0.7363 ← best
  α=0.6 | mAP@5=0.7594 | P@1=0.7848 | R@5=0.7259
  α=0.7 | mAP@5=0.7319 | P@1=0.7600 | R@5=0.6986
  α=0.8 | mAP@5=0.7003 | P@1=0.7339 | R@5=0.6641
  α=0.9 | mAP@5=0.6729 | P@1=0.7115 | R@5=0.6393
────────────────────────────────────────────────────────────────

🏆 BEST_ALPHA_2 = 0.5  (Val mAP@5 = 0.7706)

📊 Bảng val đầy đủ:
 alpha    mAP@5  Precision@1  Recall@5
   0.1 0.629927     0.679270  0.621097
   0.2 0.665432     0.712409  0.650657
   0.3 0.713904     0.754307  0.692314
   0.4 0.754786     0.784088  0.725988
   0.5 0.770575  

## 🧪 Cell 10: Đánh giá TEST SET (Chạy 1 lần duy nhất!)

In [11]:

# Sửa None thành 0.5
print(f'🧪 Đánh giá TEST SET với BEST_ALPHA_2 = {best_alpha_2:.1f}')
#best_alpha_2, best_map5_2 = 0.5, -1.0
print('⚠️  Đây là lần chạy DUY NHẤT trên test set!\n')

test_metrics_2 = evaluate_retrieval_clip(
    query_df    = df_test,
    gallery_df  = df_gallery,
    query_img   = test_img_feats,
    query_txt   = test_txt_feats,
    gallery_img = gallery_img_feats,
    gallery_txt = gallery_txt_feats,
    alpha       = best_alpha_2, K=5
)

print(f'📊 KẾT QUẢ — Baseline 2 ({MODEL_LABEL}) trên TEST SET:')
print(f'   mAP@5        = {test_metrics_2["mAP@5"]:.4f}')
print(f'   Precision@1  = {test_metrics_2["Precision@1"]:.4f}')
print(f'   Recall@5     = {test_metrics_2["Recall@5"]:.4f}')

🧪 Đánh giá TEST SET với BEST_ALPHA_2 = 0.5
⚠️  Đây là lần chạy DUY NHẤT trên test set!

📊 KẾT QUẢ — Baseline 2 (MobileCLIP (mobileclip_s0)) trên TEST SET:
   mAP@5        = 0.7708
   Precision@1  = 0.7937
   Recall@5     = 0.7430


## 📋 Cell 11: XUẤT RA FILE

In [12]:
import pandas as pd, os

# ─── Tổng hợp metrics ─────────────────────────────────────────────────────────
metrics_data = [
    {
        "Model (Phương pháp chính)": "MobileCLIP (Zero-Shot Fusion)",
        "Kích thước Vector (Dim)": str(embed_dim),
        "Alpha tối ưu (Validation)": round(float(best_alpha_2), 1),
        "Test mAP@5": round(test_metrics_2["mAP@5"], 4),
        "Test Precision@1": round(test_metrics_2["Precision@1"], 4),
        "Test Recall@5": round(test_metrics_2["Recall@5"], 4),
    }
]

df_metrics = pd.DataFrame(metrics_data)
print('📊 Final Metrics:')
print(df_metrics.to_string(index=False))

# ─── Lưu local (Colab /content/) ──────────────────────────────────────────────
LOCAL_CSV = "/content/final_metric.csv"
df_metrics.to_csv(LOCAL_CSV, index=False, encoding='utf-8-sig')
print(f"\n✅ Đã lưu local : {LOCAL_CSV}")

# ─── Lưu lên Google Drive (cùng thư mục DATA_DIR) ─────────────────────────────
DRIVE_CSV = os.path.join(DATA_DIR, "final_metric.csv")
df_metrics.to_csv(DRIVE_CSV, index=False, encoding='utf-8-sig')
print(f"✅ Đã lưu Drive  : {DRIVE_CSV}")

print('\n🏁 Hoàn tất xuất kết quả!')


📊 Final Metrics:
    Model (Phương pháp chính) Kích thước Vector (Dim)  Alpha tối ưu (Validation)  Test mAP@5  Test Precision@1  Test Recall@5
MobileCLIP (Zero-Shot Fusion)                     512                        0.5      0.7708            0.7937          0.743

✅ Đã lưu local : /content/final_metric.csv
✅ Đã lưu Drive  : /content/drive/MyDrive/DuLieuPython/final_metric.csv

🏁 Hoàn tất xuất kết quả!


---
# 🚀 PIPELINE 2 GIAI ĐOẠN: MobileCLIP + DINOv2 Re-ranking

**Kiến trúc:**
- **Giai đoạn 1 (GĐ1):** MobileCLIP lấy top-K ứng viên (K = 100)
- **Giai đoạn 2 (GĐ2):** DINOv2-Base tính lại similarity → sắp xếp lại → Top-5 cuối cùng

**Mục tiêu:** Nâng cấp baseline MobileCLIP + Alpha Tuning (mAP@5 ≈ 0.77)

## 🧠 Bước 0: Chuẩn bị – Lưu các thành phần MobileCLIP đã có

In [13]:
import os
import numpy as np
import faiss

# ── Đường dẫn lưu cache features ──────────────────────────────
FEAT_DIR = '/content/features'
os.makedirs(FEAT_DIR, exist_ok=True)

# ── Lưu cache gallery features MobileCLIP (nếu chưa có) ───────
GALLERY_IMG_CACHE = os.path.join(FEAT_DIR, 'mobileclip_gallery_img.npy')
GALLERY_TXT_CACHE = os.path.join(FEAT_DIR, 'mobileclip_gallery_txt.npy')

np.save(GALLERY_IMG_CACHE, gallery_img_feats)
np.save(GALLERY_TXT_CACHE, gallery_txt_feats)
print(f'✅ Đã lưu gallery features MobileCLIP:')
print(f'   img: {GALLERY_IMG_CACHE}  shape={gallery_img_feats.shape}')
print(f'   txt: {GALLERY_TXT_CACHE}  shape={gallery_txt_feats.shape}')

# ── Build FAISS index cho GĐ1 ─────────────────────────────────
# Dùng best_alpha_2 từ bước grid search đã chạy
gallery_fused_stage1 = fuse_and_normalize_clip(
    gallery_img_feats, gallery_txt_feats, best_alpha_2
)
faiss_index_stage1 = build_faiss_index(gallery_fused_stage1)

print(f'\n✅ FAISS index GĐ1 sẵn sàng, best_alpha_2 = {best_alpha_2:.1f}')
print(f'   Gallery size: {faiss_index_stage1.ntotal:,} vectors, dim={gallery_fused_stage1.shape[1]}')

✅ Đã lưu gallery features MobileCLIP:
   img: /content/features/mobileclip_gallery_img.npy  shape=(34250, 512)
   txt: /content/features/mobileclip_gallery_txt.npy  shape=(34250, 512)

✅ FAISS index GĐ1 sẵn sàng, best_alpha_2 = 0.5
   Gallery size: 34,250 vectors, dim=512


## 🦕 Bước 1: Cache Đặc Trưng DINOv2-Base Cho Gallery

Chạy **1 lần duy nhất** – kết quả được lưu vào `features/dinov2_gallery.npy`.

> **Lưu ý quản lý VRAM:** DINOv2 và MobileCLIP không thể đồng thời trên GPU T4.  
> Hãy chắc chắn **xóa MobileCLIP** trước khi load DINOv2 (xem Bước 1.5).

In [14]:
import torch
import torch.nn.functional as F
from torchvision import transforms
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import gc, os

DINO_CACHE_PATH = os.path.join(FEAT_DIR, 'dinov2_gallery.npy')

if not os.path.exists(DINO_CACHE_PATH):
    print('🔄 Đang tính toán DINOv2 gallery features (lần đầu tiên)...')
    print('⚠️  Dọn VRAM trước khi load DINOv2...')

    # ── 1.5: Giải phóng MobileCLIP khỏi VRAM ──────────────────
    try:
        del clip_model
        print('   clip_model đã xóa')
    except NameError:
        pass
    gc.collect()
    torch.cuda.empty_cache()
    print('🧹 VRAM đã dọn sạch!')

    # ── 1.1: Load DINOv2-Base ─────────────────────────────────
    print('\n🦕 Đang load DINOv2-Base...')
    dinov2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
    dinov2 = dinov2.to(DEVICE).eval()
    print('✅ DINOv2-Base sẵn sàng!')

    # ── 1.2: Transform ảnh chuẩn DINOv2 ───────────────────────
    transform_dino = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    @torch.no_grad()
    def get_dinov2_embedding(img_path):
        """Trả về vector 768 chiều đã chuẩn hóa L2."""
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224), (128, 128, 128))
        img_tensor = transform_dino(img).unsqueeze(0).to(DEVICE)
        feats = dinov2.forward_features(img_tensor)['x_norm_clstoken']
        feats = F.normalize(feats, dim=-1)
        return feats.cpu().numpy().flatten()

    # ── 1.3: Tính và lưu cache ─────────────────────────────────
    gallery_paths = [
        os.path.join(IMG_DIR, fname) for fname in df_gallery['image']
    ]
    gallery_dino = []
    for path in tqdm(gallery_paths, desc='🦕 DINOv2 gallery features'):
        gallery_dino.append(get_dinov2_embedding(path))
    gallery_dino = np.array(gallery_dino, dtype='float32')
    np.save(DINO_CACHE_PATH, gallery_dino)
    print(f'\n✅ Đặc trưng DINOv2 gallery: {gallery_dino.shape}')
    print(f'   Đã lưu: {DINO_CACHE_PATH}')

else:
    print(f'✅ Load DINOv2 gallery cache từ file...')
    gallery_dino = np.load(DINO_CACHE_PATH)

    # Cần load lại DINOv2 model cho inference query
    if 'dinov2' not in globals() or globals()['dinov2'] is None:
        print('🔄 Đang load lại DINOv2-Base model...')
        try:
            del clip_model
        except NameError:
            pass
        gc.collect()
        torch.cuda.empty_cache()
        dinov2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
        dinov2 = dinov2.to(DEVICE).eval()

        transform_dino = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

        @torch.no_grad()
        def get_dinov2_embedding(img_path):
            """Trả về vector 768 chiều đã chuẩn hóa L2."""
            try:
                img = Image.open(img_path).convert('RGB')
            except Exception:
                img = Image.new('RGB', (224, 224), (128, 128, 128))
            img_tensor = transform_dino(img).unsqueeze(0).to(DEVICE)
            feats = dinov2.forward_features(img_tensor)['x_norm_clstoken']
            feats = F.normalize(feats, dim=-1)
            return feats.cpu().numpy().flatten()

print(f'\n✅ DINOv2 gallery shape: {gallery_dino.shape}')

In [ ]:
# Chạy cell này 1 lần để force recompute với vitb14
import os
for cache_path in [
    os.path.join(FEAT_DIR, 'dinov2_gallery.npy'),
    os.path.join(FEAT_DIR, 'dinov2_val.npy'),
    os.path.join(FEAT_DIR, 'dinov2_test.npy'),
]:
    if os.path.exists(cache_path):
        os.remove(cache_path)
        print(f'🗑️  Đã xóa cache cũ: {cache_path}')
print('✅ Sẵn sàng recompute với DINOv2-Base')

In [ ]:
# ── Bổ sung: Extract DINOv2 features theo batch cho tập query ──────────────
@torch.no_grad()
def extract_dinov2_features_batch(df_input, img_dir, batch_size=64):
    """Trích xuất DINOv2 features theo batch — nhanh hơn gọi từng ảnh ~20-50x."""
    paths = [os.path.join(img_dir, fname) for fname in df_input['image']]
    all_feats = []

    for i in tqdm(range(0, len(paths), batch_size), desc='🦕 DINOv2 batch features'):
        batch_imgs = []
        for p in paths[i:i + batch_size]:
            try:
                img = Image.open(p).convert('RGB')
            except Exception:
                img = Image.new('RGB', (224, 224), (128, 128, 128))
            batch_imgs.append(transform_dino(img))

        batch_tensor = torch.stack(batch_imgs).to(DEVICE)
        feats = dinov2.forward_features(batch_tensor)['x_norm_clstoken']
        feats = F.normalize(feats, dim=-1)
        all_feats.append(feats.cpu().float().numpy())

    return np.vstack(all_feats)


# Tính và cache val + test DINOv2 features
VAL_DINO_CACHE  = os.path.join(FEAT_DIR, 'dinov2_val.npy')
TEST_DINO_CACHE = os.path.join(FEAT_DIR, 'dinov2_test.npy')

if not os.path.exists(VAL_DINO_CACHE):
    print('⏳ Đang tính DINOv2 val features...')
    val_dino_feats = extract_dinov2_features_batch(df_val, IMG_DIR)
    np.save(VAL_DINO_CACHE, val_dino_feats)
    print(f'✅ Đã lưu: {VAL_DINO_CACHE}  shape={val_dino_feats.shape}')
else:
    val_dino_feats = np.load(VAL_DINO_CACHE)
    print(f'✅ Loaded val DINOv2 cache: {val_dino_feats.shape}')

if not os.path.exists(TEST_DINO_CACHE):
    print('⏳ Đang tính DINOv2 test features...')
    test_dino_feats = extract_dinov2_features_batch(df_test, IMG_DIR)
    np.save(TEST_DINO_CACHE, test_dino_feats)
    print(f'✅ Đã lưu: {TEST_DINO_CACHE}  shape={test_dino_feats.shape}')
else:
    test_dino_feats = np.load(TEST_DINO_CACHE)
    print(f'✅ Loaded test DINOv2 cache: {test_dino_feats.shape}')

## 🔎 Bước 2: Hàm Tìm Kiếm 2 Giai Đoạn

In [26]:
# Xây dựng mapping index: posting_id -> index trong gallery
gallery_pids_list = df_gallery['posting_id'].tolist()


def search_mobileclip_stage1(query_fused_vec, top_k=100):
    """
    GĐ1: Dùng FAISS để tìm top_k ứng viên.
    query_fused_vec: np.array shape (dim,) đã fusion + normalize.
    Trả về: (candidate_indices, candidate_scores)
    """
    q = query_fused_vec.reshape(1, -1).astype(np.float32)
    scores, indices = faiss_index_stage1.search(q, top_k)
    return indices[0].tolist(), scores[0].tolist()


def fuse_single_clip(img_feat, txt_feat, alpha):
    """Fusion + L2-normalize cho 1 query."""
    fused = alpha * img_feat + (1 - alpha) * txt_feat
    norm = np.linalg.norm(fused)
    if norm < 1e-10:
        norm = 1e-10
    return (fused / norm).astype(np.float32)


def search_two_stage(query_img_feat, query_txt_feat, query_dino_feat,
                     alpha, query_idx=None, retrieval_k=100, final_k=5,
                     beta=0.3):
    """
    Pipeline 2 giai đoạn — dùng pre-computed DINOv2 feature thay vì load ảnh từng lần.
    Đã bỏ tham số query_img_path (không cần nữa).
    """
    # ─── Giai đoạn 1: MobileCLIP FAISS ──────────────────────────────────────
    q_fused = fuse_single_clip(query_img_feat, query_txt_feat, alpha)
    raw_indices, raw_scores = search_mobileclip_stage1(q_fused, top_k=retrieval_k + 1)

    candidate_indices, candidate_clip_scores = [], []
    for idx, score in zip(raw_indices, raw_scores):
        if idx != query_idx:
            candidate_indices.append(idx)
            candidate_clip_scores.append(score)
        if len(candidate_indices) == retrieval_k:
            break

    # ─── Giai đoạn 2: DINOv2 Re-ranking (dùng pre-computed vector) ──────────
    gallery_dino_cands = gallery_dino[candidate_indices]          # (retrieval_k, dim)
    dino_scores = (query_dino_feat @ gallery_dino_cands.T).tolist()  # vectorized dot product

    # ─── Fusion điểm ─────────────────────────────────────────────────────────
    combined  = beta * np.array(dino_scores) + (1 - beta) * np.array(candidate_clip_scores)
    new_order = np.argsort(combined)[::-1]
    final_indices = [candidate_indices[i] for i in new_order[:final_k]]
    final_scores  = [combined[i]          for i in new_order[:final_k]]
    return final_indices, final_scores

print('✅ Hàm search_two_stage (vectorized) sẵn sàng!')

## 🎛️ Bước 3: Hàm Đánh Giá + Tuning `retrieval_k` Trên Validation Set

> **Lưu ý:** Chỉ tuning trên **Val set** – KHÔNG dùng Test set để tuning!

In [27]:
from tqdm.notebook import tqdm


def evaluate_map_two_stage(query_df, alpha, retrieval_k, final_k=5,
                           query_img_feats=None, query_txt_feats=None,
                           query_dino_feats=None,   # ← tham số mới
                           beta=0.3):   # ← thêm beta
    """
    Đánh giá mAP@final_k với pipeline 2 giai đoạn.
    Args:
        query_df       : DataFrame (val hoặc test)
        alpha          : trọng số fusion MobileCLIP
        retrieval_k    : số ứng viên lấy từ GĐ1
        final_k        : số kết quả đánh giá cuối cùng
        query_img_feats: np.array đặc trưng ảnh MobileCLIP của query
        query_txt_feats: np.array đặc trưng text MobileCLIP của query
        beta           : trọng số cho DINOv2 trong fusion (0 = chỉ CLIP, 1 = chỉ DINOv2)
    Returns:
        dict chứa mAP@5, Precision@1, Recall@5
    """
    # Ánh xạ posting_id → index trong gallery (để lấy query_idx)
    gallery_pids = df_gallery['posting_id'].tolist()
    pid_to_gallery_idx = {pid: idx for idx, pid in enumerate(gallery_pids)}

    gt_dict = get_ground_truth_dict(df_gallery)
    ap_list, p1_list, r5_list = [], [], []

    for i, row in tqdm(
        enumerate(query_df.itertuples()),
        total=len(query_df),
        desc=f'🔍 Eval 2-stage (k={retrieval_k}, β={beta:.2f})'
    ):
        qid = row.posting_id
        relevant = gt_dict.get(qid, set()) - {qid}
        if not relevant:
            continue

        q_img_feat  = query_img_feats[i]
        q_txt_feat  = query_txt_feats[i]
        q_dino_feat = query_dino_feats[i]           # ← dùng pre-computed
        query_idx  = pid_to_gallery_idx[qid]   # index của ảnh query trong gallery

        try:
            final_indices, _ = search_two_stage(
                query_img_feat  = q_img_feat,
                query_txt_feat  = q_txt_feat,
                query_dino_feat = q_dino_feat,      # ← truyền vào
                alpha           = alpha,
                query_idx       = query_idx,
                retrieval_k     = retrieval_k,
                final_k         = final_k,
                beta            = beta,
            )
        except Exception as e:
            ap_list.append(0.0); p1_list.append(0.0); r5_list.append(0.0)
            continue

        # Chuyển index sang posting_id (không cần lọc self‑match nữa)
        retrieved_pids = [gallery_pids[idx] for idx in final_indices]

        hits, ap = 0, 0.0
        for rank, pid in enumerate(retrieved_pids, 1):
            if pid in relevant:
                hits += 1
                ap   += hits / rank
        ap_list.append(ap / min(len(relevant), final_k))
        p1_list.append(1.0 if (retrieved_pids and retrieved_pids[0] in relevant) else 0.0)
        r5_list.append(len(set(retrieved_pids) & relevant) / len(relevant))

    return {
        'mAP@5': float(np.mean(ap_list)),
        'Precision@1': float(np.mean(p1_list)),
        'Recall@5': float(np.mean(r5_list)),
    }


# ── Tuning retrieval_k trên VAL SET ───────────────────────────
best_k_dino     = 100
best_val_map_dino = 0.0
k_results_dino  = []

print(f'🎛️  Tuning retrieval_k với best_alpha_2 = {best_alpha_2:.1f}')
print('─' * 60)

for k in [50, 100, 150, 200]:
    # Bổ sung đặc trưng của tập val
    val_m = evaluate_map_two_stage(
        query_df=df_val,
        alpha=best_alpha_2,
        retrieval_k=k,
        final_k=5,
        query_img_feats=val_img_feats,   # ← bắt buộc
        query_txt_feats=val_txt_feats,   # ← bắt buộc
        query_dino_feats=val_dino_feats,  # ← thêm dòng này
    )
    k_results_dino.append({'retrieval_k': k, **val_m})
    marker = ' ← BEST' if val_m['mAP@5'] > best_val_map_dino else ''
    print(
        f'  retrieval_k={k:3d} | '
        f'mAP@5={val_m["mAP@5"]:.4f} | '
        f'P@1={val_m["Precision@1"]:.4f} | '
        f'R@5={val_m["Recall@5"]:.4f}{marker}'
    )
    if val_m['mAP@5'] > best_val_map_dino:
        best_val_map_dino = val_m['mAP@5']
        best_k_dino       = k

print('─' * 60)
print(f'\n✅ Chọn retrieval_k = {best_k_dino} với Val mAP@5 = {best_val_map_dino:.4f}')

# Bảng đầy đủ
import pandas as pd
df_k_results = pd.DataFrame(k_results_dino)
print('\n📊 Bảng đầy đủ:')
print(df_k_results.to_string(index=False))


In [28]:
best_beta = 0.3
best_val_map_beta = 0.0

print(f'🎛️  Tuning beta với best_alpha_2={best_alpha_2:.1f}, retrieval_k={best_k_dino}')
print('─' * 60)

for beta in np.arange(0.2, 0.6, 0.05):   # 0.2, 0.25, 0.3, ..., 0.55
    val_m = evaluate_map_two_stage(
        query_df=df_val,
        alpha=best_alpha_2,
        retrieval_k=best_k_dino,
        final_k=5,
        query_img_feats=val_img_feats,
        query_txt_feats=val_txt_feats,
        query_dino_feats=val_dino_feats,   # ← thêm dòng này
        beta=beta
    )
    print(f'beta={beta:.2f} | mAP@5={val_m["mAP@5"]:.4f} | P@1={val_m["Precision@1"]:.4f} | R@5={val_m["Recall@5"]:.4f}')
    if val_m['mAP@5'] > best_val_map_beta:
        best_val_map_beta = val_m['mAP@5']
        best_beta = beta

print(f'\n✅ Best beta = {best_beta:.2f} với Val mAP@5 = {best_val_map_beta:.4f}')

## 🧪 Bước 4: Đánh Giá CUỐI CÙNG Trên Test Set (Chạy 1 Lần!)

> ⚠️ **CHỈ CHẠY CELL NÀY 1 LẦN DUY NHẤT** sau khi đã hoàn tất tuning trên Val set.

In [29]:
import pandas as pd

print('⚠️  Đánh giá TEST SET – CHỈ CHẠY 1 LẦN DUY NHẤT!')
print(f'   best_alpha_2 = {best_alpha_2:.1f}')
print(f'   best_k_dino  = {best_k_dino}')
print(f'   best_beta    = {best_beta:.2f}')
print('─' * 60)

test_metrics_dino = evaluate_map_two_stage(
    query_df=df_test,
    alpha=best_alpha_2,
    retrieval_k=best_k_dino,
    final_k=5,
    query_img_feats=test_img_feats,
    query_txt_feats=test_txt_feats,
    query_dino_feats=test_dino_feats,  # ← thêm dòng này
    beta=best_beta
)

print(f'\n🏆 KẾT QUẢ CUỐI CÙNG – MobileCLIP + DINOv2 Re-ranking (Fusion):')
print(f'   mAP@5        = {test_metrics_dino["mAP@5"]:.4f}')
print(f'   Precision@1  = {test_metrics_dino["Precision@1"]:.4f}')
print(f'   Recall@5     = {test_metrics_dino["Recall@5"]:.4f}')

# So sánh với baseline
if 'test_metrics_2' in globals():
    improvement = test_metrics_dino['mAP@5'] - test_metrics_2['mAP@5']
    print(f'\n📈 So sánh với Baseline MobileCLIP:')
    print(f'   Baseline mAP@5      = {test_metrics_2["mAP@5"]:.4f}')
    print(f'   2-Stage mAP@5       = {test_metrics_dino["mAP@5"]:.4f}')
    print(f'   Cải thiện (Δ mAP@5) = {improvement:+.4f}')
else:
    print('\n⚠️  Chưa có kết quả baseline MobileCLIP để so sánh. Hãy chạy lại cell đánh giá baseline.')

## 💾 Bước 5: Xuất Kết Quả Ra File CSV

In [30]:
import pandas as pd, os

# ── Tổng hợp tất cả metrics ───────────────────────────────────
metrics_data_full = [
    {
        'Method'          : 'MobileCLIP (Zero-Shot Fusion)',
        'Dim'             : str(embed_dim),
        'Alpha'           : round(float(best_alpha_2), 1),
        'Retrieval_K'     : '-',
        'Test_mAP@5'      : round(test_metrics_2['mAP@5'], 4),
        'Test_Precision@1': round(test_metrics_2['Precision@1'], 4),
        'Test_Recall@5'   : round(test_metrics_2['Recall@5'], 4),
    },
    {
        'Method'          : 'MobileCLIP + DINOv2 Re-ranking',
        'Dim'             : f'{embed_dim}+768',
        'Alpha'           : round(float(best_alpha_2), 1),
        'Retrieval_K'     : best_k_dino,
        'Test_mAP@5'      : round(test_metrics_dino['mAP@5'], 4),
        'Test_Precision@1': round(test_metrics_dino['Precision@1'], 4),
        'Test_Recall@5'   : round(test_metrics_dino['Recall@5'], 4),
    },
]

df_final = pd.DataFrame(metrics_data_full)
print('📊 Final Metrics Table:')
print(df_final.to_string(index=False))

# ── Lưu local (/content/) ─────────────────────────────────────
LOCAL_CSV_DINO = '/content/final_metric_dinov2.csv'
df_final.to_csv(LOCAL_CSV_DINO, index=False, encoding='utf-8-sig')
print(f'\n✅ Đã lưu local : {LOCAL_CSV_DINO}')

# ── Lưu lên Google Drive ──────────────────────────────────────
DRIVE_CSV_DINO = os.path.join(DATA_DIR, 'final_metric_dinov2.csv')
df_final.to_csv(DRIVE_CSV_DINO, index=False, encoding='utf-8-sig')
print(f'✅ Đã lưu Drive  : {DRIVE_CSV_DINO}')

print('\n🎉 Hoàn tất pipeline MobileCLIP + DINOv2 Re-ranking!')